# chess-gnn: self-play GIF

Play one self-play game with a trained checkpoint and render it as an animated GIF,
one frame per ply. Optionally overlay the top-k predicted-move arrows for each position.

Uses the same `python-chess` SVG board renderer as `train_sl.ipynb`, then rasterizes
those SVG frames with `rsvg-convert` or `magick`, so Cairo is not required.

In [ ]:
# --- environment setup ---------------------------------------------------
import sys, subprocess, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules
print("colab:", IN_COLAB)

REPO_URL = ""  # optional: set to your GitHub URL to auto-clone on Colab
REPO_DIR = pathlib.Path("/content/chess") if IN_COLAB else pathlib.Path.cwd().parent

if IN_COLAB:
    if REPO_URL and not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    elif not REPO_DIR.exists():
        from google.colab import files  # type: ignore
        up = files.upload()
        name = next(iter(up))
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["unzip", "-q", name, "-d", str(REPO_DIR)])

    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "librsvg2-bin"])
    subprocess.check_call([
        "pip", "install", "-q",
        "torch", "torch-geometric", "python-chess", "zstandard",
        "Pillow",
    ])
    subprocess.check_call(["pip", "install", "-q", "-e", str(REPO_DIR)])
else:
    try:
        import PIL  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow"])

# Make the package importable when running locally from notebooks/
src_path = str((REPO_DIR / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

SVG_RENDERER = shutil.which("rsvg-convert") or shutil.which("magick")
assert SVG_RENDERER, (
    "Install `rsvg-convert` (recommended) or `magick` to rasterize SVG frames "
    "without Cairo."
)
print("svg renderer:", SVG_RENDERER)

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

In [ ]:
# --- load a checkpoint ---------------------------------------------------
# Point CKPT at any trained model (SL or RL). `load_model` reads the arch
# config from the file (or infers it from weights for pre-config ckpts).

CKPT = REPO_DIR / "checkpoints" / "sl" / "sl_final.pt"
# CKPT = REPO_DIR / "checkpoints" / "rl" / "rl_final.pt"

assert CKPT.exists(), f"No checkpoint at {CKPT}."
print("using:", CKPT)

from chess_gnn.model import load_model
model = load_model(CKPT, device=DEVICE)
print("arch:", model.config)

In [ ]:
# --- play one self-play game and record per-ply boards + rankings --------
# We replay the recorded trajectory through the agent to recover the
# ranking for each position (so we can overlay arrows).

import chess
from chess_gnn.selfplay import play_self_game
from chess_gnn.play import GNNAgent
from chess_gnn.moves import decode_move

TEMPERATURE = 0.05   # self-play sampling temperature
MAX_PLIES   = 200   # cap for the game
MCTS_SIMS   = 64   # PUCT sims for the arrow overlay; set to 0 for raw policy (much faster)
SEED        = 0     # set for reproducibility; None = random each run

if SEED is not None:
    torch.manual_seed(SEED)

traj = play_self_game(model, device=DEVICE, temperature=TEMPERATURE, max_plies=MAX_PLIES)
print(f"result: {traj.result} | plies: {len(traj.actions)}")

# Rebuild the sequence of boards (one per ply, plus the terminal position)
# and the predicted-move rankings for each non-terminal position.
# Note: with MCTS_SIMS=200, ranking ~60 positions takes a while.
agent = GNNAgent(
    model, device=DEVICE,
    default_temperature=TEMPERATURE,
    num_simulations=MCTS_SIMS,
)
boards = [chess.Board()]
rankings = []
moves_played = []
b = chess.Board()
for action in traj.actions:
    rankings.append(agent.rank_moves(b))
    mv = decode_move(b, action)
    moves_played.append(mv)
    b.push(mv)
    boards.append(b.copy())
print(f"frames: {len(boards)} (1 initial + {len(moves_played)} after each move)")

In [ ]:
# --- render each frame and assemble the GIF ------------------------------
# Use the same `python-chess` SVG board renderer as `train_sl.ipynb`, then
# rasterize those SVG frames for the GIF without relying on Cairo.

import io
import subprocess
import chess.svg
from PIL import Image

from chess_gnn.viz import _color_for_prob

TOPK        = 5                # overlay top-k predicted moves
FRAME_SIZE  = 480              # pixels per side
FRAME_MS    = 450              # per-frame duration
FINAL_MS    = 2500             # hold the last frame longer
OUT_GIF     = REPO_DIR / "docs" / "selfplay.gif"
OUT_GIF.parent.mkdir(parents=True, exist_ok=True)


def svg_to_png(svg: str, size: int) -> Image.Image:
    svg_bytes = svg.encode("utf-8")
    if SVG_RENDERER.endswith("rsvg-convert"):
        proc = subprocess.run(
            [SVG_RENDERER, "-w", str(size), "-h", str(size), "-f", "png"],
            input=svg_bytes,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
    else:
        proc = subprocess.run(
            [SVG_RENDERER, "svg:-", "-resize", f"{size}x{size}", "png:-"],
            input=svg_bytes,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
    return Image.open(io.BytesIO(proc.stdout)).convert("RGB")


svg_frames = []
frames = []
for i, board in enumerate(boards):
    svg_arrows = []
    if i < len(rankings):
        rk = rankings[i]
        k = min(TOPK, len(rk.moves))
        if k > 0:
            max_p = max(rk.probabilities[:k]) or 1.0
            for mv, p in zip(rk.moves[:k], rk.probabilities[:k]):
                color = _color_for_prob(p / max_p)
                svg_arrows.append(chess.svg.Arrow(mv.from_square, mv.to_square, color=color))

    last_move = moves_played[i - 1] if i > 0 else None
    if last_move is not None:
        svg_arrows.append(chess.svg.Arrow(last_move.from_square, last_move.to_square,
                                          color="#1fa84aff"))

    svg = chess.svg.board(board, arrows=svg_arrows, lastmove=last_move, size=FRAME_SIZE)
    svg_frames.append(svg)
    frames.append(svg_to_png(svg, FRAME_SIZE))

durations = [FRAME_MS] * len(frames)
durations[-1] = FINAL_MS
frames[0].save(
    OUT_GIF,
    save_all=True,
    append_images=frames[1:],
    duration=durations,
    loop=0,
    optimize=False,
    disposal=2,
)
print(f"wrote {OUT_GIF} ({OUT_GIF.stat().st_size / 1e6:.2f} MB, {len(frames)} frames)")

In [ ]:
# --- preview in the notebook ---------------------------------------------
from IPython.display import Image as IPyImage, display

display(IPyImage(filename=str(OUT_GIF)))
print(f"gif saved to {OUT_GIF}")